# TRABALHO FINAL INTEGRADOR - SEMANA 4
## Inteligencia Computacional I: construcao do modelo inteligente
### Cenario 4: Deteccao de Fraude ou Transacoes Financeiras Suspeitas

---

**Disciplina:** Topicos Especiais 2 - Agrupamento de Dados e Inteligencia Computacional  
**Etapa:** Semana 4 de 5  
**Produto esperado:** modelo inteligente funcional em versao inicial

---

## Objetivo da Semana

Construir um modelo supervisionado inicial para deteccao de fraudes, usando Random Forest, e comparar dois cenarios:

- modelo sem clusters, usando apenas as variaveis originais da base;
- modelo com clusters, usando as variaveis originais mais os sinais gerados na Semana 3.

A comparacao segue a estrategia definida nas semanas anteriores: verificar se os agrupamentos agregam valor ao modelo de Inteligencia Computacional.

## 1. Definicao da tarefa de IC

A tarefa escolhida e uma **classificacao binaria supervisionada**.

- `Class = 0`: transacao legitima;
- `Class = 1`: transacao fraudulenta.

Como a base e extremamente desbalanceada, a acuracia nao sera usada como metrica principal. O foco sera em AUC-ROC, Average Precision, Recall, Precision, F1-Score e matriz de confusao.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "scripts"))

from semana_4_modelo_ic import (  # noqa: E402
    BASE_FEATURES,
    CLUSTER_FEATURES,
    OUTPUT_DIR,
    evaluate_model,
    load_week4_dataset,
)

## 2. Carregamento e integracao dos dados

A base original `dados/creditcard.csv` e integrada com `dados/saida_clusters_semana3.csv` por meio da coluna `original_index`. Essa juncao garante que cada transacao receba os rotulos de cluster corretos.

In [ ]:
dataset = load_week4_dataset()

print(f"Registros: {len(dataset):,}".replace(",", "."))
print(f"Fraudes: {int(dataset['Class'].sum())}")
print(f"Taxa de fraude: {dataset['Class'].mean():.4%}")

dataset.head()

## 3. Preparacao das features

Foram preparados dois conjuntos de atributos:

- **sem clusters:** `Time`, `Amount` e `V1` a `V28`;
- **com clusters:** variaveis originais + `cluster_kmeans_semana3` + `dbscan_ruido`.

As variaveis de cluster foram tratadas como categoricas e convertidas por one-hot encoding.

In [ ]:
y = dataset["Class"].astype(int)

x_without_clusters = dataset[BASE_FEATURES]

x_with_clusters = pd.get_dummies(
    dataset[BASE_FEATURES + CLUSTER_FEATURES].astype(
        {"cluster_kmeans_semana3": "category", "dbscan_ruido": "category"}
    ),
    columns=CLUSTER_FEATURES,
    drop_first=False,
    dtype=int,
)

print("Features sem clusters:", x_without_clusters.shape[1])
print("Features com clusters:", x_with_clusters.shape[1])
x_with_clusters.head()

## 4. Escolha da tecnica

A tecnica escolhida foi **Random Forest**, com `class_weight='balanced'`.

A escolha segue o planejamento da pasta `instrucoes_IC/` porque Random Forest:

- lida bem com relacoes nao lineares;
- funciona como baseline supervisionado robusto;
- permite penalizar mais a classe minoritaria;
- facilita a comparacao entre modelo puro e modelo enriquecido por clusters.

## 5. Treinamento dos modelos

Nesta etapa sao treinados dois modelos Random Forest. A funcao `evaluate_model` aplica divisao treino/teste estratificada e validacao cruzada estratificada com 5 folds.

In [ ]:
results = [
    evaluate_model("Random Forest sem clusters", x_without_clusters, y),
    evaluate_model("Random Forest com clusters", x_with_clusters, y),
]

metrics = pd.DataFrame(results)
metrics

## 6. Avaliacao preliminar no conjunto de teste

In [ ]:
test_columns = [
    "modelo",
    "auc_roc_teste",
    "average_precision_teste",
    "recall_teste",
    "precision_teste",
    "f1_teste",
    "tn",
    "fp",
    "fn",
    "tp",
]

metrics[test_columns]

## 7. Validacao cruzada estratificada

A validacao cruzada reduz a dependencia de um unico split treino/teste. Isso e especialmente importante porque a amostra enriquecida possui apenas 25 fraudes.

In [ ]:
cv_columns = [
    "modelo",
    "auc_roc_cv_media",
    "auc_roc_cv_desvio",
    "avg_precision_cv_media",
    "avg_precision_cv_desvio",
    "recall_cv_media",
    "precision_cv_media",
    "f1_cv_media",
]

metrics[cv_columns]

## 8. Comparacao inicial com e sem clusters

A comparacao inicial mostra um resultado misto:

- no conjunto de teste, os dois modelos tiveram o mesmo Recall, Precision e F1;
- no conjunto de teste, o modelo com clusters teve Average Precision ligeiramente maior;
- na validacao cruzada, o modelo sem clusters teve AUC-ROC media e Average Precision media melhores;
- portanto, nesta amostra, os clusters ainda nao comprovaram ganho consistente no classificador supervisionado.

Essa conclusao e preliminar porque a saida de clusters da Semana 3 contem 15.000 registros e apenas 25 fraudes. Na Semana 5, o ideal e ampliar a avaliacao, ajustar hiperparametros e analisar limiares de decisao.

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
output_path = OUTPUT_DIR / "semana_4_metricas_modelo_ic.csv"
metrics.to_csv(output_path, index=False)
output_path

## 9. Conclusao da Semana 4

A Semana 4 entregou um modelo inteligente funcional em versao inicial. A tarefa de IC foi definida, a tecnica foi escolhida e justificada, os dados foram preparados, o modelo foi treinado e a avaliacao preliminar foi realizada.

O produto gerado e um classificador Random Forest para deteccao de fraudes, comparado nos cenarios sem clusters e com clusters. A proxima etapa sera consolidar o desempenho com uma avaliacao mais ampla e ajustes de modelo.